# 01 — EDA e Qualidade

Este notebook faz a inspeção inicial do dataset: shape, tipos, nulos, cardinalidade e distribuições de preço, desconto e rating. Também exporta um relatório de qualidade para `reports/`.

In [1]:
from __future__ import annotations

from pathlib import Path
import sys

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

PROJECT_DIR = Path.cwd().resolve()
while PROJECT_DIR.name != 'amazon-product-intelligence' and PROJECT_DIR.parent != PROJECT_DIR:
    PROJECT_DIR = PROJECT_DIR.parent

DATA_RAW = PROJECT_DIR / 'data' / 'raw' / 'dados_amazon.csv'
REPORTS_DIR = PROJECT_DIR / 'reports'
FIGURES_DIR = REPORTS_DIR / 'figures'
REPORTS_DIR.mkdir(parents=True, exist_ok=True)
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

sys.path.insert(0, str(PROJECT_DIR / 'notebooks'))
from src.preprocessing import load_raw_dataset

sns.set_theme(style='whitegrid')


C:\Users\flavi\Anaconda3\Lib\site-packages\pandas\core\computation\expressions.py:22: UserWarning: Pandas requires version '2.10.2' or newer of 'numexpr' (version '2.10.1' currently installed).
  from pandas.core.computation.check import NUMEXPR_INSTALLED


## Carregamento e inspeção geral

In [2]:
df_raw = load_raw_dataset(DATA_RAW)
df_raw.shape


(1465, 16)

In [3]:
df_raw.head(3)


,product_id,product_name,category,discounted_price,actual_price,discount_percentage,rating,rating_count,about_product,user_id,user_name,review_id,review_title,review_content,img_link,product_link
0,B07JW9H4J1,Wayona Nylon Braided USB to Lightning Fast Cha...,Computers&Accessories|Accessories&Peripherals|...,₹399,"₹1,099",64%,4.2,"24,269",High Compatibility : Compatible With iPhone 12...,"AG3D6O4STAQKAY2UVGEUV46KN35Q,AHMY5CWJMMK5BJRBB...","Manav,Adarsh gupta,Sundeep,S.Sayeed Ahmed,jasp...","R3HXWT0LRP0NMF,R2AJM3LFTLZHFO,R6AQJGUP6P86,R1K...","Satisfied,Charging is really fast,Value for mo...",Looks durable Charging is fine tooNo complains...,https://m.media-amazon.com/images/W/WEBP_40237...,https://www.amazon.in/Wayona-Braided-WN3LG1-Sy...
1,B098NS6PVG,Ambrane Unbreakable 60W / 3A Fast Charging 1.5...,Computers&Accessories|Accessories&Peripherals|...,₹199,₹349,43%,4.0,"43,994","Compatible with all Type C enabled devices, be...","AECPFYFQVRUWC3KGNLJIOREFP5LQ,AGYYVPDD7YG7FYNBX...","ArdKn,Nirbhay kumar,Sagar Viswanathan,Asp,Plac...","RGIQEG07R9HS2,R1SMWZQ86XIN8U,R2J3Y1WL29GWDE,RY...","A Good Braided Cable for Your Type C Device,Go...",I ordered this cable to connect my phone to An...,https://m.media-amazon.com/images/W/WEBP_40237...,https://www.amazon.in/Ambrane-Unbreakable-Char...
2,B096MSW6CT,Sounce Fast Phone Charging Cable & Data Sync U...,Computers&Accessories|Accessories&Peripherals|...,₹199,"₹1,899",90%,3.9,"7,928",【 Fast Charger& Data Sync】-With built-in safet...,"AGU3BBQ2V2DDAMOAKGFAWDDQ6QHA,AESFLDV2PT363T2AQ...","Kunal,Himanshu,viswanath,sai niharka,saqib mal...","R3J3EQQ9TZI5ZJ,R3E7WBGK7ID0KV,RWU79XKQ6I1QF,R2...","Good speed for earlier versions,Good Product,W...","Not quite durable and sturdy,https://m.media-a...",https://m.media-amazon.com/images/W/WEBP_40237...,https://www.amazon.in/Sounce-iPhone-Charging-C...


In [4]:
dtypes = df_raw.dtypes.astype(str).to_frame('dtype')
dtypes


,dtype
product_id,str
product_name,str
category,str
discounted_price,str
actual_price,str
discount_percentage,str
rating,str
rating_count,str
about_product,str
user_id,str


## Completude e missingness

In [5]:
missing_pct = (df_raw.isna().mean() * 100).sort_values(ascending=False)
missing_pct.to_frame('missing_%').head(20)


,missing_%
rating_count,0.136519
product_id,0.000000
product_name,0.000000
category,0.000000
discounted_price,0.000000
actual_price,0.000000
discount_percentage,0.000000
rating,0.000000
about_product,0.000000
user_id,0.000000


In [6]:
plt.figure(figsize=(12, 6))
sns.heatmap(df_raw.isna(), cbar=False)
plt.title('Missingness Map (row × column)')
plt.tight_layout()
plt.savefig(FIGURES_DIR / 'missingness_map.png', dpi=160)
plt.close()


## Distribuições principais

In [7]:
plt.figure(figsize=(10, 4))
sns.histplot(df_raw['rating'].dropna(), bins=30, kde=True)
plt.title('Distribution: rating')
plt.tight_layout()
plt.savefig(FIGURES_DIR / 'dist_rating.png', dpi=160)
plt.close()


In [8]:
def _clean_currency(s: pd.Series) -> pd.Series:
    return (
        s.astype(str)
        .str.replace('₹', '', regex=False)
        .str.replace(',', '', regex=False)
        .replace({'nan': np.nan, 'None': np.nan, '': np.nan})
        .astype(float)
    )

discounted = _clean_currency(df_raw['discounted_price'])
actual = _clean_currency(df_raw['actual_price'])
discount_pct = (
    df_raw['discount_percentage']
    .astype(str)
    .str.replace('%', '', regex=False)
    .str.replace(',', '', regex=False)
    .replace({'nan': np.nan, 'None': np.nan, '': np.nan})
    .astype(float)
)

plt.figure(figsize=(10, 4))
sns.histplot(discounted.dropna(), bins=40, kde=False)
plt.title('Distribution: discounted_price (cleaned)')
plt.tight_layout()
plt.savefig(FIGURES_DIR / 'dist_discounted_price.png', dpi=160)
plt.close()

plt.figure(figsize=(10, 4))
sns.histplot(discount_pct.dropna(), bins=40, kde=True)
plt.title('Distribution: discount_percentage (cleaned)')
plt.tight_layout()
plt.savefig(FIGURES_DIR / 'dist_discount_pct.png', dpi=160)
plt.close()


## Distribuição por categoria

In [9]:
main_category = df_raw['category'].astype(str).str.split('|').str[0].replace({'nan': np.nan})
cat_counts = main_category.value_counts().head(15)
cat_counts


category
Electronics              526
Computers&Accessories    453
Home&Kitchen             448
OfficeProducts            31
MusicalInstruments         2
HomeImprovement            2
Toys&Games                 1
Car&Motorbike              1
Health&PersonalCare        1
Name: count, dtype: int64

In [10]:
plt.figure(figsize=(12, 5))
sns.barplot(x=cat_counts.index, y=cat_counts.values)
plt.xticks(rotation=45, ha='right')
plt.title('Top Categories (main_category)')
plt.tight_layout()
plt.savefig(FIGURES_DIR / 'products_by_category.png', dpi=160)
plt.close()


## Exportar relatório de qualidade

In [11]:
critical_cols = ['product_id', 'product_name', 'category', 'discounted_price', 'actual_price', 'discount_percentage', 'rating', 'rating_count']
critical_missing = (df_raw[critical_cols].isna().mean() * 100).sort_values(ascending=False)

report_lines = []
report_lines.append('# Data Quality Report')
report_lines.append('')
report_lines.append(f'- Rows: {len(df_raw):,}')
report_lines.append(f'- Columns: {df_raw.shape[1]}')
report_lines.append('')
report_lines.append('## Missingness (Top 20)')
for col, pct in missing_pct.head(20).items():
    report_lines.append(f'- {col}: {pct:.2f}%')
report_lines.append('')
report_lines.append('## Critical Fields Missingness')
for col, pct in critical_missing.items():
    report_lines.append(f'- {col}: {pct:.2f}%')

(REPORTS_DIR / 'data_quality_report.md').write_text("\\n".join(report_lines), encoding='utf-8')
REPORTS_DIR / 'data_quality_report.md'


WindowsPath('C:/Users/flavi/Documents/GitHub/Amazon_Product_Clustering/amazon-product-intelligence/reports/data_quality_report.md')